# FARM-SATHI: Soil Classification Model Training Pipeline

This notebook provides an end-to-end, reproducible training and evaluation pipeline for the **FARM-SATHI Soil Analyzer**.
It uses MobileNetV2 transfer learning on the **Comprehensive Soil Classification Dataset** (Farhan Sheth et al., Elsevier 2025) covering 7 distinct soil categories:
- Alluvial Soil
- Arid Soil
- Black Soil
- Laterite Soil
- Mountain Soil
- Red Soil
- Yellow Soil


## 1. Environment Setup
Install and import required deep learning, data processing, and evaluation packages.


In [ ]:
import os
import sys
import json
import random
import hashlib
from datetime import datetime, timezone
from pathlib import Path

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")


## 2. Dataset Download or Google Drive Mounting
Clone the verified soil dataset repository from GitHub.


In [ ]:
!git clone --depth 1 https://github.com/Phantom-fs/Soil-Classification-Dataset.git soil_repo
DATASET_ROOT = Path("soil_repo/Orignal-Dataset")
print(f"Dataset root directory: {DATASET_ROOT}")


## 3. Dataset Directory Verification
Verify that all 7 class folders are present and accessible.


In [ ]:
SOIL_CLASS_MAP = {
    "Alluvial_Soil": "Alluvial Soil",
    "Arid_Soil": "Arid Soil",
    "Black_Soil": "Black Soil",
    "Laterite_Soil": "Laterite Soil",
    "Mountain_Soil": "Mountain Soil",
    "Red_Soil": "Red Soil",
    "Yellow_Soil": "Yellow Soil",
}

verified_folders = {}
for folder_name, display_name in SOIL_CLASS_MAP.items():
    folder_path = DATASET_ROOT / folder_name
    if folder_path.is_dir():
        verified_folders[display_name] = folder_path
    else:
        print(f"Warning: Directory missing for {display_name}: {folder_path}")

print(f"Successfully verified {len(verified_folders)} soil class folders.")


## 4. Image Count per Class
Inspect the number of genuine photographic images per category.


In [ ]:
class_counts = {}
for class_name, folder in verified_folders.items():
    images = [f for f in folder.glob("*.*") if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    class_counts[class_name] = len(images)
    print(f"{class_name:18s}: {len(images)} images")

total_images = sum(class_counts.values())
print(f"\nTotal Soil Images: {total_images}")


## 5. Dataset Visualization
Plot the class frequency distribution to understand representation.


In [ ]:
plt.figure(figsize=(10, 5))
classes = sorted(list(class_counts.keys()))
counts = [class_counts[c] for c in classes]
colors = ['#8D6E63', '#D7CCC8', '#3E2723', '#BF360C', '#4E342E', '#D32F2F', '#FBC02D']

bars = plt.bar(classes, counts, color=colors, edgecolor='#333', linewidth=1)
plt.title("FARM-SATHI Soil Dataset - Class Distribution", fontsize=14, pad=15)
plt.xlabel("Soil Category", fontsize=12)
plt.ylabel("Image Count", fontsize=12)
plt.xticks(rotation=25, ha='right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 3, int(yval), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


## 6. Sample Soil Image Display
Visualize representative soil photographs from each verified category.


In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(20, 4))

for idx, class_name in enumerate(classes):
    folder = verified_folders[class_name]
    sample_img_path = next(folder.glob("*.*"))
    img = Image.open(sample_img_path).convert("RGB")
    
    axes[idx].imshow(img)
    axes[idx].set_title(class_name, fontsize=11, fontweight='bold')
    axes[idx].axis("off")

plt.suptitle("Representative Soil Samples by Class", fontsize=15, y=1.05)
plt.tight_layout()
plt.show()


## 7. Invalid Image Detection
Scan every file in the dataset to detect corrupted headers, zero-byte files, or unreadable formats.


In [ ]:
valid_samples = []
corrupted_files = []

for folder_name, display_name in SOIL_CLASS_MAP.items():
    folder_path = DATASET_ROOT / folder_name
    for img_path in folder_path.glob("*.*"):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        try:
            with Image.open(img_path) as im:
                im.verify()
            valid_samples.append((str(img_path), display_name))
        except Exception as e:
            corrupted_files.append((str(img_path), str(e)))

print("Integrity verification complete:")
print(f"  - Valid readable images: {len(valid_samples)}")
print(f"  - Corrupted/unreadable files: {len(corrupted_files)}")


## 8. Duplicate Detection where Practical
Compute MD5 checksums of image binary contents to identify exact duplicates.


In [ ]:
hashes = {}
duplicates = []

for path_str, class_name in valid_samples:
    with open(path_str, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash in hashes:
        duplicates.append((path_str, hashes[file_hash]))
    else:
        hashes[file_hash] = path_str

print(f"Identified {len(duplicates)} exact duplicate files out of {len(valid_samples)} images.")
if duplicates:
    print(f"Example duplicate: {duplicates[0][0]} matches {duplicates[0][1]}")


## 9. Train, Validation, and Test Splitting
Perform a stratified 70% / 15% / 15% split to prevent class imbalance and avoid data leakage.


In [ ]:
all_paths = [s[0] for s in valid_samples]
all_labels_names = [s[1] for s in valid_samples]

class_to_idx = {name: i for i, name in enumerate(classes)}
all_label_indices = [class_to_idx[name] for name in all_labels_names]

# 70% Train, 30% Temp (Val + Test)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_label_indices, test_size=0.30, random_state=SEED, stratify=all_label_indices
)

# 15% Validation, 15% Test
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels
)

print("Stratified dataset splits:")
print(f"  - Training set:   {len(train_paths):4d} images ({len(train_paths)/len(all_paths)*100:.1f}%)")
print(f"  - Validation set: {len(val_paths):4d} images ({len(val_paths)/len(all_paths)*100:.1f}%)")
print(f"  - Held-out Test:  {len(test_paths):4d} images ({len(test_paths)/len(all_paths)*100:.1f}%)")


## 10. Image Resizing and Normalization
Define high-performance `tf.data` pipelines resizing images to 224x224 RGB.


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

def make_dataset(paths, labels, training=False):
    paths_ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        paths_ds = paths_ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    
    def parse_image(path, label):
        raw = tf.io.read_file(path)
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        img.set_shape([IMAGE_SIZE, IMAGE_SIZE, 3])
        return img, label

    ds = paths_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)
print("Data pipelines prepared successfully.")


## 11. Data Augmentation
Introduce realistic field variations: random rotation, horizontal flip, zoom, and contrast adjustment.


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.08, seed=SEED),
    tf.keras.layers.RandomZoom(0.10, seed=SEED),
    tf.keras.layers.RandomContrast(0.08, seed=SEED),
], name="field_augmentation")


## 12. Transfer Learning Model Creation
Construct MobileNetV2 with pre-trained ImageNet weights, embedded normalization, and custom classifier head.


In [ ]:
def create_soil_model(num_classes):
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), dtype=tf.float32, name="rgb_0_255")
    
    # 1. Augmentation
    augmented = data_augmentation(inputs)
    
    # 2. Preprocessing (-1 to 1 MobileNetV2 standard)
    preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(augmented)
    
    # 3. Backbone
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    backbone.trainable = False
    
    # 4. Classification Head
    features = backbone(preprocessed, training=False)
    pooled = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(features)
    dropped = tf.keras.layers.Dropout(0.25, name="dropout")(pooled)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="soil_probabilities")(dropped)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="farm_sathi_soil_mobilenetv2")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = create_soil_model(len(classes))
model.summary()


## 13. Model Training
Train the model on the training split using the Adam optimizer.


In [ ]:
EPOCHS = 6
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]

print(f"Commencing training for {EPOCHS} epochs...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


## 14. Validation
Evaluate performance across epochs on the validation set.


In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Best Validation Loss:     {val_loss:.4f}")
print(f"Best Validation Accuracy: {val_acc*100:.2f}%")


## 15. Accuracy and Loss Graphs
Plot learning curves for training and validation subsets.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy plot
ax1.plot(epochs_range, history.history['accuracy'], label="Training Accuracy", marker='o')
ax1.plot(epochs_range, history.history['val_accuracy'], label="Validation Accuracy", marker='s')
ax1.set_title("Training and Validation Accuracy", fontsize=13)
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Accuracy", fontsize=11)
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.4)

# Loss plot
ax2.plot(epochs_range, history.history['loss'], label="Training Loss", marker='o', color='orange')
ax2.plot(epochs_range, history.history['val_loss'], label="Validation Loss", marker='s', color='red')
ax2.set_title("Training and Validation Loss", fontsize=13)
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Cross-Entropy Loss", fontsize=11)
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


## 16. Confusion Matrix
Evaluate class-by-class confusion on the held-out test dataset.


In [ ]:
test_probs = model.predict(test_ds, verbose=0)
test_preds = np.argmax(test_probs, axis=-1)

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix on Held-Out Test Set", fontsize=14, pad=15)
plt.xlabel("Predicted Soil Class", fontsize=12)
plt.ylabel("True Soil Class", fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 17. Classification Report
Generate per-class precision, recall, and F1-score report on unseen test data.


In [ ]:
report_str = classification_report(test_labels, test_preds, target_names=classes, digits=4)
print("HELD-OUT TEST SET CLASSIFICATION REPORT:\n")
print(report_str)


## 18. Precision, Recall, and F1-Score
Compute macro-averaged metrics summarizing overall generalization.


In [ ]:
test_acc = accuracy_score(test_labels, test_preds)
macro_prec = precision_score(test_labels, test_preds, average="macro", zero_division=0)
macro_rec = recall_score(test_labels, test_preds, average="macro", zero_division=0)
macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)

print(f"Overall Held-Out Test Metrics ({len(test_labels)} samples):")
print(f"  - Accuracy:        {test_acc * 100:.2f}%")
print(f"  - Macro Precision: {macro_prec * 100:.2f}%")
print(f"  - Macro Recall:    {macro_rec * 100:.2f}%")
print(f"  - Macro F1-Score:  {macro_f1 * 100:.2f}%")


## 19. Test Sample Predictions
Inspect model predictions and confidence probabilities on random test samples.


In [ ]:
sample_indices = random.sample(range(len(test_paths)), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    img = Image.open(test_paths[idx]).convert("RGB")
    true_cls = classes[test_labels[idx]]
    pred_cls = classes[test_preds[idx]]
    conf = test_probs[idx][test_preds[idx]] * 100
    
    is_correct = (true_cls == pred_cls)
    title_color = "green" if is_correct else "red"
    
    axes[i].imshow(img)
    axes[i].set_title(f"Pred: {pred_cls} ({conf:.1f}%)\nTrue: {true_cls}",
                      color=title_color, fontsize=10, fontweight='bold')
    axes[i].axis("off")

plt.suptitle("Model Predictions on Unseen Test Samples", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 20. Model Export
Export the trained model to native Keras format (`soil_visual_model.keras`).


In [ ]:
OUTPUT_DIR = Path("farm_sathi_soil_export")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "soil_visual_model.keras"
model.save(model_path)
print(f"Model exported successfully to {model_path} ({os.path.getsize(model_path)/(1024*1024):.2f} MB)")


## 21. Label JSON Generation
Export the exact ordered list of class labels to `soil_labels.json`.


In [ ]:
labels_path = OUTPUT_DIR / "soil_labels.json"
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(classes, f, indent=2)

print(f"Labels saved to {labels_path}:")
print(json.dumps(classes, indent=2))


## 22. Model Metadata Generation
Generate `model_metadata.json` documenting the dataset, training hyperparameters, and test evaluation scores.


In [ ]:
metadata = {
    "status": "trained",
    "model_name": "MobileNetV2 Soil Classifier",
    "model_version": "1.0.0",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "architecture": "MobileNetV2 transfer learning (ImageNet initialization)",
    "input_size": [IMAGE_SIZE, IMAGE_SIZE],
    "color_mode": "RGB",
    "preprocessing": "embedded_mobilenet_v2",
    "class_count": len(classes),
    "classes": classes,
    "training_dataset": {
        "name": "Comprehensive Soil Classification Dataset",
        "citation": "Farhan Sheth et al., Engineering Applications of Artificial Intelligence (Elsevier, 2025)",
        "license": "CC-BY-NC-4.0",
        "total_images": len(valid_samples),
        "classes": classes,
        "class_distribution": class_counts
    },
    "evaluation_metrics": {
        "test_accuracy": round(float(test_acc), 4),
        "test_precision_macro": round(float(macro_prec), 4),
        "test_recall_macro": round(float(macro_rec), 4),
        "test_f1_macro": round(float(macro_f1), 4),
        "test_sample_count": len(test_labels)
    },
    "confidence_threshold": 0.45,
    "top_prediction_margin_threshold": 0.08
}

metadata_path = OUTPUT_DIR / "model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata successfully saved to {metadata_path}")


## 23. Download Trained Model Files
Bundle exported files into a zip archive for installation into FARM-SATHI (`ai-service/models/soil/`).


In [ ]:
import shutil
archive_path = shutil.make_archive("farm_sathi_soil_model_export", "zip", OUTPUT_DIR)
print(f"Export archive created: {archive_path}")

try:
    from google.colab import files
    files.download(archive_path)
    print("Download triggered in Google Colab.")
except ImportError:
    print(f"Ready for local copy at: {archive_path}")
